# EXTRACTING DATA FROM KAGGLE

In [1]:
!pip install kagglehub

from pathlib import Path
import kagglehub
import pandas as pd

#Dowwnloading the files from kaggle 

path = Path(kagglehub.dataset_download('olistbr/brazilian-ecommerce'))

#Loading the Files 

orders       = pd.read_csv(path / "olist_orders_dataset.csv")
order_items  = pd.read_csv(path / "olist_order_items_dataset.csv")
payments     = pd.read_csv(path / "olist_order_payments_dataset.csv")
reviews      = pd.read_csv(path / "olist_order_reviews_dataset.csv")
products     = pd.read_csv(path / "olist_products_dataset.csv")
customers    = pd.read_csv(path / "olist_customers_dataset.csv")
sellers      = pd.read_csv(path / "olist_sellers_dataset.csv")
geolocation  = pd.read_csv(path / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(path / "product_category_name_translation.csv")

print(orders.shape, order_items.shape, products.shape)

(99441, 8) (112650, 7) (32951, 9)


In [2]:
print(payments.shape,reviews.shape,customers.shape,sellers.shape,geolocation.shape,category_translation.shape)

(103886, 5) (99224, 7) (99441, 5) (3095, 4) (1000163, 5) (71, 2)


# Transform Phase

## ORDERS

In [3]:
orders.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [4]:
orders.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00


In [5]:
date_cols = ['order_purchase_timestamp', 
             'order_approved_at', 'order_delivered_carrier_date', 
             'order_delivered_customer_date', 'order_estimated_delivery_date']
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

In [6]:
orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [7]:
orders[date_cols].isna().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [8]:
orders.loc[orders['order_delivered_customer_date'].isna(), 'order_status']

6           invoiced
44           shipped
103         invoiced
128       processing
154          shipped
            ...     
99283       canceled
99313     processing
99347       canceled
99348    unavailable
99415    unavailable
Name: order_status, Length: 2965, dtype: object

## ORDER ITEMS

In [9]:
order_items.dtypes

order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

In [10]:
order_items.head(2)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93


In [11]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])


In [12]:
order_items.dtypes

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

In [13]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

## Payments

In [14]:
payments.dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [15]:
payments.head(5)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [16]:
payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [17]:
payments.shape

(103886, 5)

In [18]:
payments['order_id'].value_counts().head(10)

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
fedcd9f7ccdc8cba3a18defedd1a5547    19
ee9ca989fc93ba09a6eddc250ce01742    19
4bfcba9e084f46c8e3cb49b0fa6e6159    15
21577126c19bf11a0b91592e5844ba78    15
3c58bffb70dcf45f12bdf66a3c215905    14
4689b1816de42507a7d63a4617383c59    14
Name: count, dtype: int64

In [76]:
payments.loc[payments['order_id'] == 'fa65dad1b0e818e3ccc5cb0e39231352']


,order_id,payment_sequential,payment_type,payment_installments,payment_value
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
24879,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
32519,fa65dad1b0e818e3ccc5cb0e39231352,11,voucher,1,4.03


In [20]:
payments.duplicated(subset=["order_id", "payment_sequential"]).sum()

np.int64(0)

## REVIEWS

In [21]:
reviews.head(5)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [22]:
reviews.dtypes

review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

In [23]:
reviews.shape

(99224, 7)

In [24]:
reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [25]:
reviews_date_cols = ['review_answer_timestamp', 'review_creation_date'] 
reviews[reviews_date_cols] = reviews[reviews_date_cols] .apply (pd.to_datetime)

In [26]:
reviews.dtypes

review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

In [27]:
reviews['order_id'].duplicated().sum()

np.int64(551)

In [28]:
reviews['order_id'].value_counts().head(5)

order_id
c88b1d1b157a9999ce368f218a407141    3
8e17072ec97ce29f0e1f111e598b0c85    3
df56136b8031ecd28e200bb18e6ddb2e    3
03c939fd7fd3b38f8485a0f95798f1f6    3
5cb890a68b91b6158d69257e4e2bc359    2
Name: count, dtype: int64

In [29]:
reviews.loc[reviews['order_id'] == 'c88b1d1b157a9999ce368f218a407141']

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,2017-07-22,2017-07-26 13:41:07
82525,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-22,2017-07-26 13:40:22
89360,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-21,2017-07-26 13:45:15


## PRODUCTS

In [30]:
products.dtypes

product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [31]:
products.head(5)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [32]:
products.shape

(32951, 9)

In [33]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [34]:
products[products["product_weight_g"].isnull()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
products[products["product_category_name"].isna()].isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [36]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')

In [37]:
products['product_category_name'].isna().sum()

np.int64(0)

## CUSTOMERS

In [38]:
customers.dtypes

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

In [39]:
customers.shape

(99441, 5)

In [40]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

## SELLERS

In [41]:
sellers.dtypes

seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

In [42]:
sellers.shape

(3095, 4)

In [43]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [44]:
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)

## CUSTOMERS

In [45]:
customers.dtypes

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix    object
customer_city               object
customer_state              object
dtype: object

In [46]:
sellers.dtypes

seller_id                 object
seller_zip_code_prefix    object
seller_city               object
seller_state              object
dtype: object

In [47]:
customers['customer_zip_code_prefix'].str.len().value_counts()
sellers['seller_zip_code_prefix'].str.len().value_counts()

seller_zip_code_prefix
5    3095
Name: count, dtype: int64

## GEOLOCATION

In [48]:
geolocation.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

In [49]:
geolocation.head(5)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [50]:
geolocation.shape

(1000163, 5)

In [51]:
geolocation.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [52]:
geolocation['geolocation_zip_code_prefix'] = geolocation['geolocation_zip_code_prefix'].astype(str).str.zfill(5)

In [53]:
geolocation['geolocation_zip_code_prefix'].str.len().value_counts()

geolocation_zip_code_prefix
5    1000163
Name: count, dtype: int64

In [54]:
geolocation['geolocation_zip_code_prefix'].nunique()

19015

In [55]:
geolocation['geolocation_zip_code_prefix'].duplicated().sum()

np.int64(981148)

In [56]:
geolocation_clean = (
    geolocation.groupby('geolocation_zip_code_prefix', as_index=False)
    .agg({
        'geolocation_lat': 'mean',
        'geolocation_lng': 'mean',
        'geolocation_city': 'first',
        'geolocation_state': 'first',
    })
)

geolocation_clean.shape

(19015, 5)

In [57]:
geolocation_clean.head(5)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550190,-46.634024,sao paulo,SP
1,01002,-23.548146,-46.634979,sao paulo,SP
2,01003,-23.548994,-46.635731,sao paulo,SP
3,01004,-23.549799,-46.634757,sao paulo,SP
4,01005,-23.549456,-46.636733,sao paulo,SP


## CATEGORY TRANSLATION

In [58]:
category_translation.dtypes

product_category_name            object
product_category_name_english    object
dtype: object

In [59]:
category_translation.head(5)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [60]:
category_translation.shape

(71, 2)

In [61]:
category_translation.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [62]:
# Merging the duplicates reviews in just 1 review per order

In [63]:
reviews_agg = (
    reviews
    .sort_values('review_creation_date', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
)

In [64]:
reviews_agg.shape

(98673, 7)

In [65]:
reviews.groupby('order_id')['review_creation_date'].nunique().value_counts()

review_creation_date
1    98281
2      390
3        2
Name: count, dtype: int64

In [66]:
reviews_agg = (
    reviews
    .sort_values(["review_creation_date", "review_answer_timestamp"], ascending=False)
    .drop_duplicates(subset="order_id", keep="first")
)

reviews_agg.shape

(98673, 7)

In [67]:
## MERGING PRODUCTION WITH CATEGORY TRANSLATION TO GET ENGLISH CATEGORY NAMES

In [68]:
products = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

In [69]:
products.shape

(32951, 10)

In [70]:
products[products['product_category_name_english'].isna()]['product_category_name'].value_counts()

product_category_name
unknown                                          610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64

In [71]:
products["product_category_name_english"] = products["product_category_name_english"].fillna(
    products["product_category_name"]
)

products["product_category_name_english"].isna().sum()

np.int64(0)

In [72]:
products.shape

(32951, 10)

In [ ]:
#aggregating the duplicate rows in payments table

In [73]:
payments.head(5)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [82]:
payments.shape

(103886, 5)

In [78]:
payments_agg = (payments
               .sort_values('payment_value', ascending=False)
               .groupby('order_id')
               .agg(
                   payment_value=('payment_value', 'sum'),
                   payment_installments=('payment_installments', 'sum'),
                   payment_type=('payment_type', 'first'),
                   payment_count=('payment_type', 'size')
               )
                .reset_index())

In [79]:
payments_agg.head(5)

,order_id,payment_value,payment_installments,payment_type,payment_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,credit_card,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,credit_card,1


In [81]:
payments_agg.shape

(99440, 5)

In [83]:
payments_agg.loc[payments_agg['order_id']=='fa65dad1b0e818e3ccc5cb0e39231352']

,order_id,payment_value,payment_installments,payment_type,payment_count
97261,fa65dad1b0e818e3ccc5cb0e39231352,457.99,29,voucher,29


In [84]:
#Joining and merging our table together

In [85]:
fact = order_items.merge(orders, on='order_id', how='left')
fact.shape

(112650, 14)

In [86]:
fact=fact.merge(products, on='product_id', how='left')
fact.shape

(112650, 23)

In [88]:
fact=fact.merge(sellers, on='seller_id', how='left')
fact.shape

(112650, 26)

In [89]:
fact=fact.merge(customers, on='customer_id', how='left')
fact.shape

(112650, 30)

In [91]:
fact=fact.merge(payments_agg, on='order_id', how='left')
fact.shape

(112650, 34)

In [93]:
fact.loc[fact["payment_value"].isna()]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,payment_value,payment_installments,payment_type,payment_count,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
84389,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,...,NaN,NaN,NaN,NaN,6916ca4502d6d3bfd39818759d55d536,1.0,NaN,nao recebi o produto e nem resposta da empresa,2016-10-06,2016-10-07 18:32:28
84390,bfbd0f9bdef84302105ad712db648a6c,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,...,NaN,NaN,NaN,NaN,6916ca4502d6d3bfd39818759d55d536,1.0,NaN,nao recebi o produto e nem resposta da empresa,2016-10-06,2016-10-07 18:32:28
84391,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,...,NaN,NaN,NaN,NaN,6916ca4502d6d3bfd39818759d55d536,1.0,NaN,nao recebi o produto e nem resposta da empresa,2016-10-06,2016-10-07 18:32:28


In [92]:
fact=fact.merge(reviews_agg, on='order_id', how='left')
fact.shape

(112650, 40)

In [94]:
fact.isna().sum().sort_values(ascending = False)

review_comment_title             99235
review_comment_message           65255
order_delivered_customer_date     2454
product_name_lenght               1603
product_photos_qty                1603
product_description_lenght        1603
order_delivered_carrier_date      1194
review_answer_timestamp            942
review_creation_date               942
review_score                       942
review_id                          942
product_width_cm                    18
product_length_cm                   18
product_weight_g                    18
product_height_cm                   18
order_approved_at                   15
payment_count                        3
payment_type                         3
payment_installments                 3
payment_value                        3
product_category_name                0
customer_city                        0
product_id                           0
seller_id                            0
shipping_limit_date                  0
price                    

In [96]:
fact.dtypes

order_id                                 object
order_item_id                             int64
product_id                               object
seller_id                                object
shipping_limit_date              datetime64[ns]
price                                   float64
freight_value                           float64
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
product_category_name                    object
product_name_lenght                     float64
product_description_lenght              float64
product_photos_qty                      float64
product_weight_g                        float64
product_length_cm                       float64
product_height_cm                       

## LOAD PHASE

In [99]:
fact.to_parquet('olist_fact_table.parquet', index = False)
fact.to_csv('olist_fact_table.csv', index = False)

print('saved:', fact.shape)

saved: (112650, 40)
